# KLTN: Tác động phi tuyến của đa dạng hóa xuất khẩu đến tăng trưởng kinh tế

Dữ liệu bảng cân bằng 128 quốc gia, 2000–2022 (2.944 quan sát), sheet `DATA` của `data/input/KLTN_du_lieu.xlsx`, tạo bằng `jupiter_notebook/tao_du_lieu.py`. Dừng ở 2022 vì HHI thị trường của World Bank chỉ công bố đến năm này. Notebook chạy đủ 9 bước trong `target/CHECKLIST_KLTN.md`, mỗi bước ra một bảng, thêm Bước 10 tái lập bài báo mẫu trên 44 nước của bài báo. Chạy **Run All** từ trên xuống; toàn bộ bảng lưu vào `data/output/KLTN_ket_qua.xlsx`, các bảng theo bố cục bài báo lưu vào `data/output/KLTN_bang_bai_bao.xlsx`.

Mô hình: `GDPG ~ EDI + CAP + FDI + GOV + LAB`, với EDI là HHI hoặc THEIL.
- GDPG: tăng trưởng GDP thực bình quân đầu người (WDI NY.GDP.PCAP.KD.ZG).
- HHI: chỉ số Herfindahl–Hirschman tập trung **thị trường** xuất khẩu (World Bank WITS), cùng khái niệm với bài mẫu.
- THEIL: chỉ số Theil tổng của UNCTAD; THEIL_PROD là thành phần theo sản phẩm, dùng ở Bước 10.

HHI/THEIL càng cao thì xuất khẩu càng **tập trung** (ít đa dạng), nên hệ số âm hàm ý đa dạng hóa hỗ trợ tăng trưởng.

Ký hiệu: `***` p<0,01; `**` p<0,05; `*` p<0,1. Sai số chuẩn trong ngoặc.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects

pd.set_option("display.width", 200, "display.max_columns", 30)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "input").exists())
INPUT = ROOT / "data" / "input" / "KLTN_du_lieu.xlsx"
OUTPUT = ROOT / "data" / "output" / "KLTN_ket_qua.xlsx"
OUTPUT_BAIBAO = ROOT / "data" / "output" / "KLTN_bang_bai_bao.xlsx"

Y = "GDPG"
EDIS = ["HHI", "THEIL"]
CONTROLS = ["CAP", "FDI", "GOV", "LAB"]
N_BOOT = 1000  # số lần bootstrap cho kiểm định ngưỡng
GRID = 200     # số điểm lưới tìm ngưỡng
TRIM = 0.05    # mỗi regime giữ tối thiểu 5% quan sát
SEED = 2024
TABLES = {}    # tên sheet -> DataFrame, ghi ra Excel ở ô cuối

In [2]:
df = pd.read_excel(INPUT, sheet_name="DATA").sort_values(["ID", "Year"]).reset_index(drop=True)
for e in EDIS:
    df[f"{e}2"] = df[e] ** 2
    for d in ["GFC", "COVID", "DEV"]:
        df[f"{e}x{d}"] = df[e] * df[d]

N, T = df["ID"].nunique(), df["Year"].nunique()
assert len(df) == N * T and not df[[Y, *EDIS, *CONTROLS]].isna().any().any(), "Panel phải cân bằng và không thiếu số liệu"
print(f"{N} quốc gia x {T} năm ({df.Year.min()}-{df.Year.max()}) = {len(df)} quan sát;",
      f"Developed: {df.loc[df.DEV == 1, 'ID'].nunique()}, Developing: {df.loc[df.DEV == 0, 'ID'].nunique()}")
df.head()

128 quốc gia x 23 năm (2000-2022) = 2944 quan sát; Developed: 37, Developing: 91


,ID,Country,Code,Year,Region,DEV,GFC,COVID,GDPG,HHI,THEIL,THEIL_PROD,CAP,FDI,GOV,LAB,HHI2,HHIxGFC,HHIxCOVID,HHIxDEV,THEIL2,THEILxGFC,THEILxCOVID,THEILxDEV
0,1,Albania,ALB,2000,Đang phát triển – Châu Âu/Trung Á,0,0,0,8.149968,0.390747,7.139283,2.278300,34.040139,3.989321,9.731558,-1.467925,0.152683,0.0,0.0,0.0,50.969362,0.0,0.0,0.0
1,1,Albania,ALB,2001,Đang phát triển – Châu Âu/Trung Á,0,0,0,9.890194,0.415648,7.284668,2.387903,38.228045,5.107089,10.593907,-1.801800,0.172763,0.0,0.0,0.0,53.066388,0.0,0.0,0.0
2,1,Albania,ALB,2002,Đang phát triển – Châu Âu/Trung Á,0,0,0,4.942623,0.564760,7.370769,2.422850,38.575336,2.990031,11.236148,-0.116632,0.318954,0.0,0.0,0.0,54.328236,0.0,0.0,0.0
3,1,Albania,ALB,2003,Đang phát triển – Châu Âu/Trung Á,0,0,0,5.728106,0.500269,7.318967,2.378376,36.722128,3.068687,11.050514,-1.059497,0.250269,0.0,0.0,0.0,53.567278,0.0,0.0,0.0
4,1,Albania,ALB,2004,Đang phát triển – Châu Âu/Trung Á,0,0,0,5.707123,0.546933,7.232003,2.328293,36.331947,4.607823,11.188577,-1.143219,0.299136,0.0,0.0,0.0,52.301867,0.0,0.0,0.0


In [3]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""


def fit(data, xs, kind="FE", cov="clustered"):
    """Ước lượng Pooled / FE / RE. cov: clustered (cluster theo quốc gia) hoặc unadjusted."""
    d = data.set_index(["ID", "Year"])
    kw = dict(cov_type="clustered", cluster_entity=True) if cov == "clustered" else dict(cov_type="unadjusted")
    if kind == "FE":
        return PanelOLS(d[Y], d[xs], entity_effects=True).fit(**kw)
    X = sm.add_constant(d[xs])
    return (PooledOLS if kind == "Pooled" else RandomEffects)(d[Y], X).fit(**kw)


def bang(models):
    """Gộp nhiều kết quả hồi quy thành một bảng: hệ số kèm sao, sai số chuẩn ở dòng dưới."""
    names = list(dict.fromkeys(v for r in models.values() for v in r.params.index))
    names = [v for v in names if v != "const"] + [v for v in names if v == "const"]
    index, cols = [], {m: [] for m in models}
    for v in names:
        index += [v, ""]
        for m, r in models.items():
            if v in r.params.index:
                cols[m] += [f"{r.params[v]:.4f}{stars(r.pvalues[v])}", f"({r.std_errors[v]:.4f})"]
            else:
                cols[m] += ["", ""]
    index += ["Số quan sát", "R2"]
    for m, r in models.items():
        cols[m] += [f"{int(r.nobs)}", f"{r.rsquared:.4f}"]
    return pd.DataFrame(cols, index=index)


def luu(ten, bang_kq):
    TABLES[ten] = bang_kq
    return bang_kq

## Bước 1. Thống kê mô tả

Xem trung bình, độ lệch chuẩn, nhỏ nhất, lớn nhất để mô tả đặc điểm mẫu.

In [4]:
VARS = [Y, *EDIS, *CONTROLS]
luu("1_ThongKeMoTa", df[VARS].describe().T[["count", "mean", "std", "min", "max"]]
    .rename(columns={"count": "Số quan sát", "mean": "Trung bình", "std": "Độ lệch chuẩn", "min": "Nhỏ nhất", "max": "Lớn nhất"}).round(4))

,Số quan sát,Trung bình,Độ lệch chuẩn,Nhỏ nhất,Lớn nhất
GDPG,2944.0,2.2966,4.6240,-55.2936,49.0739
HHI,2944.0,0.1537,0.1418,0.0303,0.8088
THEIL,2944.0,6.0339,1.5546,2.9862,10.3899
CAP,2944.0,24.2348,7.3978,1.0968,76.7823
FDI,2944.0,6.0514,25.1430,-296.0133,452.2209
GOV,2944.0,15.6256,5.0292,2.0484,36.1679
LAB,2944.0,1.6670,2.3933,-19.9788,25.8612


## Bước 2. Tương quan và đa cộng tuyến

Không cặp biến độc lập nào được tương quan quá cao (|r| > 0,8), VIF < 10. HHI và THEIL là hai thước đo thay thế nhau nên không đưa cùng lúc vào một mô hình; VIF tính riêng cho từng bộ biến.

In [5]:
luu("2_TuongQuan", df[VARS].corr().round(3))

,GDPG,HHI,THEIL,CAP,FDI,GOV,LAB
GDPG,1.000,-0.068,-0.053,0.210,0.061,-0.170,0.048
HHI,-0.068,1.000,0.579,0.007,-0.062,-0.193,0.119
THEIL,-0.053,0.579,1.000,-0.001,-0.003,-0.289,0.293
CAP,0.210,0.007,-0.001,1.000,0.023,-0.012,0.001
FDI,0.061,-0.062,-0.003,0.023,1.000,0.021,0.033
GOV,-0.170,-0.193,-0.289,-0.012,0.021,1.000,-0.173
LAB,0.048,0.119,0.293,0.001,0.033,-0.173,1.000


In [6]:
def vif(xs):
    X = sm.add_constant(df[xs])
    return pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=xs)

luu("2b_VIF", pd.DataFrame({f"Mô hình {e}": vif([e, *CONTROLS]).rename({e: "EDI"}) for e in EDIS}).round(3))

,Mô hình HHI,Mô hình THEIL
EDI,1.051,1.169
CAP,1.001,1.001
FDI,1.006,1.002
GOV,1.065,1.103
LAB,1.041,1.106


## Bước 3. Mô hình tuyến tính cơ sở: Pooled OLS, FE, RE

Sai số chuẩn cluster theo quốc gia.

In [7]:
lin = {(e, k): fit(df, [e, *CONTROLS], k) for e in EDIS for k in ["Pooled", "FE", "RE"]}
luu("3_TuyenTinh", bang({f"{e} - {k}": r for (e, k), r in lin.items()}))

,HHI - Pooled,HHI - FE,HHI - RE,THEIL - Pooled,THEIL - FE,THEIL - RE
HHI,-3.4209***,-3.1006**,-3.6338***,,,
,(0.8082),(1.5647),(0.9324),,,
CAP,0.1294***,0.1246***,0.1290***,0.1286***,0.1235***,0.1274***
,(0.0269),(0.0290),(0.0267),(0.0245),(0.0291),(0.0254)
FDI,0.0097**,0.0066,0.0086*,0.0107**,0.0070,0.0092*
,(0.0046),(0.0051),(0.0048),(0.0046),(0.0052),(0.0048)
GOV,-0.1694***,-0.4032***,-0.2020***,-0.1805***,-0.3895***,-0.2134***
,(0.0293),(0.1155),(0.0403),(0.0301),(0.1175),(0.0416)
LAB,0.0520,0.2665***,0.1553*,0.0938,0.2664***,0.1821**
,(0.0768),(0.0875),(0.0792),(0.0806),(0.0875),(0.0811)


## Bước 4. Lựa chọn mô hình panel

- **F-test** (H0: không có hiệu ứng cố định): p < 0,05 thì FE tốt hơn Pooled.
- **Breusch–Pagan LM** (H0: phương sai hiệu ứng ngẫu nhiên = 0): p < 0,05 thì RE tốt hơn Pooled.
- **Hausman** (H0: RE nhất quán): p < 0,05 thì chọn FE.

Ba kiểm định dùng ma trận hiệp phương sai thường (không robust) theo đúng giả định của kiểm định.

In [8]:
def hausman(data, xs):
    fe, re = (fit(data, xs, k, cov="unadjusted") for k in ["FE", "RE"])
    b = fe.params[xs] - re.params[xs]
    h = float(b @ np.linalg.pinv(fe.cov.loc[xs, xs] - re.cov.loc[xs, xs]) @ b)
    return h, stats.chi2.sf(h, len(xs))


def chon_mo_hinh(e):
    xs = [e, *CONTROLS]
    po, fe, re = (fit(df, xs, k, cov="unadjusted") for k in ["Pooled", "FE", "RE"])
    # Breusch-Pagan LM cho panel cân bằng
    u = po.resids
    lm = N * T / (2 * (T - 1)) * ((u.groupby(level=0).sum() ** 2).sum() / (u ** 2).sum() - 1) ** 2
    h, p_h = hausman(df, xs)
    p_f, p_lm = fe.f_pooled.pval, stats.chi2.sf(lm, 1)
    chon = "FE" if p_h < 0.05 and p_f < 0.05 else "RE" if p_lm < 0.05 else "Pooled OLS"
    return {"F-test": fe.f_pooled.stat, "p (F)": p_f, "BP LM": lm, "p (LM)": p_lm,
            "Hausman": h, "p (Hausman)": p_h, "Chọn": chon}

sel = pd.DataFrame({e: chon_mo_hinh(e) for e in EDIS}).T
luu("4_ChonMoHinh", sel)

,F-test,p (F),BP LM,p (LM),Hausman,p (Hausman),Chọn
HHI,3.499462,0.0,171.052281,0.0,131.248907,0.0,FE
THEIL,3.3838,0.0,159.047071,0.0,120.298396,0.0,FE


Các bước sau dùng FE (hiệu ứng cố định theo quốc gia) với sai số chuẩn cluster. Nếu bảng trên chọn mô hình khác FE thì đổi `kind` trong hàm `fit`.

## Bước 5. Mô hình phi tuyến bậc hai

`GDPG ~ EDI + EDI² + controls`. Xem đồng thời hệ số EDI (β₁) và EDI² (β₂): β₂ > 0 là dạng U, β₂ < 0 là U ngược. Điểm ngoặt = −β₁ / (2β₂), chỉ có nghĩa khi nằm trong khoảng giá trị thực của EDI.

In [9]:
quad = {e: fit(df, [e, f"{e}2", *CONTROLS]) for e in EDIS}
luu("5_BacHai", bang({f"{e} - FE bậc hai": r for e, r in quad.items()}))

,HHI - FE bậc hai,THEIL - FE bậc hai
HHI,-1.4868,
,(3.9250),
HHI2,-2.2792,
,(4.2897),
CAP,0.1245***,0.1223***
,(0.0291),(0.0293)
FDI,0.0067,0.0070
,(0.0051),(0.0051)
GOV,-0.4031***,-0.3946***
,(0.1157),(0.1176)


In [10]:
def diem_ngoat(r, e, data):
    b1, b2 = r.params[e], r.params[f"{e}2"]
    tp = -b1 / (2 * b2)
    lo, hi = data[e].min(), data[e].max()
    return {"β1 (EDI)": b1, "p (β1)": r.pvalues[e], "β2 (EDI²)": b2, "p (β2)": r.pvalues[f"{e}2"],
            "Dạng": "U" if b2 > 0 else "U ngược", "Điểm ngoặt": tp, "EDI min": lo, "EDI max": hi,
            "Trong khoảng dữ liệu": lo < tp < hi,
            "% quan sát dưới điểm ngoặt": round(100 * (data[e] < tp).mean(), 1),
            "Phi tuyến có ý nghĩa (5%)": r.pvalues[f"{e}2"] < 0.05}

luu("5b_DiemNgoat", pd.DataFrame({e: diem_ngoat(quad[e], e, df) for e in EDIS}).T)

,β1 (EDI),p (β1),β2 (EDI²),p (β2),Dạng,Điểm ngoặt,EDI min,EDI max,Trong khoảng dữ liệu,% quan sát dưới điểm ngoặt,Phi tuyến có ý nghĩa (5%)
HHI,-1.486769,0.704869,-2.279234,0.595236,U ngược,-0.326156,0.030321,0.80882,False,0.0,False
THEIL,4.003912,0.001372,-0.263794,0.00212,U ngược,7.589089,2.986203,10.389943,True,83.4,True


## Bước 6. Panel threshold regression (Hansen, 1999)

Biến ngưỡng là chính EDI; hệ số của EDI thay đổi theo regime, các biến kiểm soát giữ chung:

`GDPG = μᵢ + β₁·EDI·1(EDI ≤ γ) + β₂·EDI·1(EDI > γ) + θ·controls + ε`

- Ngưỡng γ tìm bằng cách tối thiểu tổng bình phương phần dư trên lưới phân vị 5%–95% của EDI.
- Kiểm định có ngưỡng hay không (H0: không có ngưỡng) bằng thống kê F với p-value bootstrap.

In [11]:
def _within(a, n, t):
    a = a.reshape(n, t, -1)
    return (a - a.mean(1, keepdims=True)).reshape(n * t, -1)


def _Q(p, g=None):
    # Cơ sở trực chuẩn của biến giải thích sau khi khử hiệu ứng cố định; g=None là mô hình không ngưỡng
    q = p["q"]
    R = q[:, None] if g is None else np.column_stack([q * (q <= g), q * (q > g)])
    return np.linalg.qr(np.hstack([_within(R, p["n"], p["t"]), p["C"]]))[0]


def _ssr(Q, Yv):
    return (Yv ** 2).sum(0) - ((Q.T @ Yv) ** 2).sum(0)


def threshold(data, e, controls=CONTROLS):
    d = data.sort_values(["ID", "Year"])
    n = d["ID"].nunique(); t = len(d) // n
    p = {"n": n, "t": t, "q": d[e].to_numpy(float), "C": _within(d[controls].to_numpy(float), n, t)}
    y = _within(d[[Y]].to_numpy(float), n, t)[:, 0]
    dof = n * (t - 1)
    grid = np.quantile(p["q"], np.linspace(TRIM, 1 - TRIM, GRID))

    Q0, Q1 = _Q(p), [_Q(p, g) for g in grid]
    s1_all = np.array([_ssr(Q, y) for Q in Q1])
    i = int(s1_all.argmin()); s1 = s1_all[i]; g = grid[i]
    F = (_ssr(Q0, y) - s1) / (s1 / dof)

    # Bootstrap: lấy mẫu lại nguyên chuỗi phần dư của từng quốc gia
    resid = (y - Q1[i] @ (Q1[i].T @ y)).reshape(n, t)
    Yb = resid[np.random.default_rng(SEED).integers(0, n, (N_BOOT, n))].reshape(N_BOOT, -1).T
    s1b = np.array([_ssr(Q, Yb) for Q in Q1]).min(0)
    pval = float(((_ssr(Q0, Yb) - s1b) / (s1b / dof) >= F).mean())

    dd = d.copy()
    dd[f"{e}_duoi_nguong"], dd[f"{e}_tren_nguong"] = dd[e] * (dd[e] <= g), dd[e] * (dd[e] > g)
    tom_tat = {"Ngưỡng (γ)": g, "% quan sát ≤ γ": round(100 * (p["q"] <= g).mean(), 1),
               "F": F, "p bootstrap": pval, "Có ngưỡng (5%)": pval < 0.05}
    return tom_tat, fit(dd, [f"{e}_duoi_nguong", f"{e}_tren_nguong", *controls])

In [12]:
thr = {e: threshold(df, e) for e in EDIS}
luu("6_Nguong", pd.DataFrame({e: thr[e][0] for e in EDIS}).T)

,Ngưỡng (γ),% quan sát ≤ γ,F,p bootstrap,Có ngưỡng (5%)
HHI,0.289022,87.3,8.712518,0.436,False
THEIL,7.689971,85.1,5.889965,0.892,False


In [13]:
luu("6b_HeSoRegime", bang({f"{e} - threshold": thr[e][1] for e in EDIS}))

,HHI - threshold,THEIL - threshold
HHI_duoi_nguong,2.9270,
,(3.4000),
HHI_tren_nguong,-1.9668,
,(1.3513),
CAP,0.1247***,0.1243***
,(0.0291),(0.0293)
FDI,0.0067,0.0071
,(0.0051),(0.0052)
GOV,-0.4013***,-0.3913***
,(0.1155),(0.1181)


`duoi_nguong` là hệ số của EDI khi EDI ≤ γ (đa dạng hóa cao), `tren_nguong` là khi EDI > γ. Chỉ diễn giải hai hệ số này khi kiểm định ngưỡng có ý nghĩa.

## Bước 7. So sánh Developed và Developing

Chạy riêng từng nhóm (tuyến tính và threshold), rồi dùng biến tương tác `EDI×DEV` trên toàn mẫu để kiểm định khác biệt hệ số có ý nghĩa thống kê hay không (DEV = 1 là nước phát triển; bản thân DEV bị hiệu ứng cố định hấp thụ).

In [14]:
GROUPS = {"Developed": df[df.DEV == 1], "Developing": df[df.DEV == 0]}
grp_lin = {(e, g): fit(d, [e, *CONTROLS]) for e in EDIS for g, d in GROUPS.items()}
luu("7_NhomNuoc_HoiQuy", bang({f"{e} - {g}": r for (e, g), r in grp_lin.items()}))

,HHI - Developed,HHI - Developing,THEIL - Developed,THEIL - Developing
HHI,11.6168*,-2.7453*,,
,(6.4039),(1.5137),,
CAP,0.2235***,0.0502,0.2191***,0.0477
,(0.0771),(0.0440),(0.0759),(0.0443)
FDI,0.0017,0.1799***,0.0018,0.1851***
,(0.0021),(0.0679),(0.0025),(0.0696)
GOV,-0.8240***,-0.3288***,-0.8333***,-0.3171**
,(0.1179),(0.1213),(0.1107),(0.1235)
LAB,0.2425*,0.2461***,0.2667**,0.2460***
,(0.1375),(0.0925),(0.1262),(0.0922)


In [15]:
grp_thr = {(e, g): threshold(d, e) for e in EDIS for g, d in GROUPS.items()}
luu("7b_NhomNuoc_Nguong", pd.DataFrame({f"{e} - {g}": r[0] for (e, g), r in grp_thr.items()}).T)

,Ngưỡng (γ),% quan sát ≤ γ,F,p bootstrap,Có ngưỡng (5%)
HHI - Developed,0.045346,10.5,5.604861,0.862,False
HHI - Developing,0.287101,83.2,5.119903,0.734,False
THEIL - Developed,4.810504,71.4,7.732336,0.594,False
THEIL - Developing,7.043182,61.1,9.500653,0.35,False


In [16]:
luu("7c_NhomNuoc_Regime", bang({f"{e} - {g}": r[1] for (e, g), r in grp_thr.items()}))

,HHI - Developed,HHI - Developing,THEIL - Developed,THEIL - Developing
HHI_duoi_nguong,-18.6778**,2.4099,,
,(9.5084),(3.5530),,
HHI_tren_nguong,10.0560,-1.7059,,
,(6.2411),(1.3716),,
CAP,0.2147***,0.0510,0.2188***,0.0462
,(0.0762),(0.0434),(0.0744),(0.0442)
FDI,0.0025,0.1796***,0.0016,0.1914***
,(0.0023),(0.0671),(0.0023),(0.0705)
GOV,-0.8256***,-0.3265***,-0.8460***,-0.3009**
,(0.1173),(0.1212),(0.1091),(0.1179)


In [17]:
inter = {e: fit(df, [e, f"{e}xDEV", *CONTROLS]) for e in EDIS}
luu("7d_TuongTacDEV", bang({f"{e} - FE tương tác DEV": r for e, r in inter.items()}))

,HHI - FE tương tác DEV,THEIL - FE tương tác DEV
HHI,-3.5413**,
,(1.6227),
HHIxDEV,17.4234**,
,(7.2860),
CAP,0.1230***,0.1216***
,(0.0290),(0.0292)
FDI,0.0072,0.0075
,(0.0051),(0.0054)
GOV,-0.4046***,-0.3969***
,(0.1154),(0.1179)


Hệ số `EDI` là tác động ở nhóm Developing; `EDIxDEV` là chênh lệch của nhóm Developed so với Developing. `EDIxDEV` có ý nghĩa thì hai nhóm khác nhau có ý nghĩa thống kê.

## Bước 8. Khủng hoảng GFC (2008–2009) và COVID (2020–2021)

`GDPG ~ EDI + EDI×GFC + EDI×COVID + GFC + COVID + controls`. Hệ số tương tác cho biết tác động của EDI thay đổi bao nhiêu trong năm khủng hoảng.

In [18]:
crisis = {e: fit(df, [e, f"{e}xGFC", f"{e}xCOVID", "GFC", "COVID", *CONTROLS]) for e in EDIS}
luu("8_KhungHoang", bang({f"{e} - FE khủng hoảng": r for e, r in crisis.items()}))

,HHI - FE khủng hoảng,THEIL - FE khủng hoảng
HHI,-3.2320**,
,(1.5314),
HHIxGFC,4.1078**,
,(1.8393),
HHIxCOVID,-2.5738,
,(1.8824),
GFC,-3.2918***,-6.1508***
,(0.4214),(1.0348)
COVID,-1.3831***,-0.0233
,(0.4066),(0.7886)


## Bước 9. Kiểm định độ vững

- Thay HHI ↔ THEIL: đã chạy song song ở mọi bước, so sánh hai cột.
- Sai số chuẩn: mọi mô hình FE ở trên đã dùng sai số chuẩn cluster theo quốc gia.
- Outlier: đếm quan sát lệch quá 3 độ lệch chuẩn, rồi chạy lại sau khi winsorize 1%–99% toàn bộ biến liên tục.

In [19]:
z = (df[VARS] - df[VARS].mean()) / df[VARS].std()
luu("9_Outlier", pd.DataFrame({"Số quan sát |z| > 3": (z.abs() > 3).sum(), "Tỷ lệ (%)": (100 * (z.abs() > 3).mean()).round(2)}))

,Số quan sát |z| > 3,Tỷ lệ (%)
GDPG,40,1.36
HHI,87,2.96
THEIL,0,0.00
CAP,34,1.15
FDI,25,0.85
GOV,8,0.27
LAB,56,1.90


In [20]:
dfw = df.copy()
dfw[VARS] = dfw[VARS].clip(dfw[VARS].quantile(0.01), dfw[VARS].quantile(0.99), axis=1)
for e in EDIS:
    dfw[f"{e}2"] = dfw[e] ** 2

rob = {}
for e in EDIS:
    for ten, xs in {"tuyến tính": [e, *CONTROLS], "bậc hai": [e, f"{e}2", *CONTROLS]}.items():
        rob[f"{e} {ten} - cluster"] = fit(df, xs)
        rob[f"{e} {ten} - winsorize"] = fit(dfw, xs)
luu("9b_DoVung", bang(rob))

,HHI tuyến tính - cluster,HHI tuyến tính - winsorize,HHI bậc hai - cluster,HHI bậc hai - winsorize,THEIL tuyến tính - cluster,THEIL tuyến tính - winsorize,THEIL bậc hai - cluster,THEIL bậc hai - winsorize
HHI,-3.1006**,-2.3087*,-1.4868,-0.8363,,,,
,(1.5647),(1.2023),(3.9250),(3.6316),,,,
CAP,0.1246***,0.1152***,0.1245***,0.1151***,0.1235***,0.1135***,0.1223***,0.1138***
,(0.0290),(0.0239),(0.0291),(0.0239),(0.0291),(0.0237),(0.0293),(0.0239)
FDI,0.0066,0.0486**,0.0067,0.0489**,0.0070,0.0506**,0.0070,0.0489**
,(0.0051),(0.0226),(0.0051),(0.0227),(0.0052),(0.0227),(0.0051),(0.0224)
GOV,-0.4032***,-0.3414***,-0.4031***,-0.3414***,-0.3895***,-0.3306***,-0.3946***,-0.3348***
,(0.1155),(0.0677),(0.1157),(0.0679),(0.1175),(0.0697),(0.1176),(0.0697)
LAB,0.2665***,0.2729***,0.2666***,0.2727***,0.2664***,0.2709***,0.2651***,0.2701***
,(0.0875),(0.0705),(0.0874),(0.0703),(0.0875),(0.0704),(0.0876),(0.0703)


## Bước 10. Tái lập bài báo mẫu (Trinh & Thuy, 2021) trên 44 nước của bài báo

Lấy đúng 44 nước ở Bảng 1 bài báo. Guyana và Jamaica không có trong mẫu 128 nước (thiếu dữ liệu WDI), nên còn **42 nước**. Chạy đúng phương pháp của bài báo:

- Threshold theo phương trình (2): **pooled**, hằng số và dummy GFC dùng chung, **mọi biến X** (EDI, CAP, FDI, GOV, LAB) đổi hệ số theo regime; trimming 15% như Bai–Perron. Báo cáo cả hai kiểm định: giá trị tới hạn Bai–Perron (như bài báo) và p-value bootstrap theo khối quốc gia (dưới H0 không ngưỡng). Bai–Perron giả định sai số độc lập; bootstrap theo cụm tính đến tương quan trong cùng quốc gia nên chặt hơn. Sai số chuẩn cluster theo quốc gia.
- Bậc hai theo phương trình (3): FE, thêm dummy GFC (không có AFC vì dữ liệu bắt đầu từ 2000).
- Hai giai đoạn: 2000–2015 (gần giai đoạn bài báo 1995–2015) và toàn giai đoạn 2000–2022. Thêm THEIL_PROD vì có thang đo gần Theil IMF của bài báo.


In [21]:
P44 = ("DZA ARG BOL BRA BGR BFA CMR CHL CHN COL CRI HRV ECU EGY SLV GEO GTM GUY HUN IND IDN JAM LTU MYS MUS "
       "MEX MDA MAR MOZ PAN PRY PER PHL POL ROU RUS SEN TZA THA TUN TUR UGA URY VNM").split()
print("Không có trong dữ liệu:", sorted(set(P44) - set(df.Code)))
df["THEIL_PROD2"] = df["THEIL_PROD"] ** 2
MAU44 = {f"2000-{n}": df[df.Code.isin(P44) & (df.Year <= n)] for n in (2015, df.Year.max())}
EDIS44 = [*EDIS, "THEIL_PROD"]


# Giá trị tới hạn Bai–Perron supF(0 vs 1), trimming 15%, q = 5 biến đổi hệ số, quy về F chia cho q (cùng thang với F ở đây).
# Nguồn: bảng Bai & Perron (2003, Econometrics Journal) trong package R mbreaks (Nguyen, Yamamoto, Perron), file R/supF/cv_3.csv: 16,14 / 18,23 / 22,40 chia 5.
CV_BP = {0.01: 22.40 / 5, 0.05: 18.23 / 5, 0.10: 16.14 / 5}
sao_bp = lambda F: "***" if F > CV_BP[0.01] else "**" if F > CV_BP[0.05] else "*" if F > CV_BP[0.10] else ""


def threshold_pooled(data, e, trim=0.15):
    """Phương trình (2) của bài báo: pooled, hằng số + GFC chung, mọi biến X đổi hệ số theo regime."""
    d = data.sort_values(["ID", "Year"]).reset_index(drop=True)
    q, y = d[e].to_numpy(float), d[Y].to_numpy(float)
    X = d[[e, *CONTROLS]].to_numpy(float)
    M0 = np.column_stack([X, np.ones(len(d)), d["GFC"]])
    k, dof = X.shape[1], len(y) - 2 * X.shape[1] - 2
    grid = np.unique(np.quantile(q, np.linspace(trim, 1 - trim, GRID)))
    D = lambda g: np.column_stack([X * (q <= g)[:, None], X * (q > g)[:, None], M0[:, -2:]])
    ssr = lambda M, v: ((v - M @ np.linalg.lstsq(M, v, rcond=None)[0]) ** 2).sum(0)

    def sup_f(v):  # v: 1 cột (dữ liệu thật) hoặc ma trận (mỗi cột một mẫu bootstrap)
        s1 = np.array([ssr(D(g), v) for g in grid])
        return (ssr(M0, v) - s1.min(0)) / k / (s1.min(0) / dof), grid[s1.argmin(0)]

    F, g = sup_f(y)
    yh = M0 @ np.linalg.lstsq(M0, y, rcond=None)[0]
    u = (y - yh).reshape(d["ID"].nunique(), -1)
    Yb = yh[:, None] + u[np.random.default_rng(SEED).integers(0, len(u), (N_BOOT, len(u)))].reshape(N_BOOT, -1).T
    pval = float((sup_f(Yb)[0] >= F).mean())

    dd = d.copy()
    for v in [e, *CONTROLS]:
        dd[f"{v}_r1"], dd[f"{v}_r2"] = dd[v] * (dd[e] <= g), dd[v] * (dd[e] > g)
    cols = [f"{v}_r{j}" for j in (1, 2) for v in [e, *CONTROLS]] + ["GFC"]
    tom_tat = {"Số nước": d["ID"].nunique(), "Ngưỡng (γ)": g, "% quan sát ≤ γ": round(100 * (q <= g).mean(), 1),
               "F": F, "p bootstrap": pval, "Có ngưỡng (5%, bootstrap)": pval < 0.05,
               "Mức ý nghĩa Bai–Perron": sao_bp(F), "Có ngưỡng (5%, Bai–Perron)": F > CV_BP[0.05]}
    return tom_tat, fit(dd, cols, "Pooled")


thr44 = {(e, m): threshold_pooled(d, e) for m, d in MAU44.items() for e in EDIS44}
luu("10_BaiBao_Nguong", pd.DataFrame({f"{e} {m}": r[0] for (e, m), r in thr44.items()}).T)


Không có trong dữ liệu: ['GUY', 'JAM']


,Số nước,Ngưỡng (γ),% quan sát ≤ γ,F,p bootstrap,"Có ngưỡng (5%, bootstrap)",Mức ý nghĩa Bai–Perron,"Có ngưỡng (5%, Bai–Perron)"
HHI 2000-2015,42,0.084037,47.0,5.196488,0.158,False,***,True
THEIL 2000-2015,42,5.064107,34.7,7.063647,0.092,False,***,True
THEIL_PROD 2000-2015,42,1.578656,32.3,9.281407,0.023,True,***,True
HHI 2000-2022,42,0.064386,26.3,5.484346,0.298,False,***,True
THEIL 2000-2022,42,5.072636,35.1,9.450108,0.034,True,***,True
THEIL_PROD 2000-2022,42,1.803324,43.2,12.766919,0.008,True,***,True


In [22]:
luu("10b_BaiBao_HeSo", bang({f"{e} {m}": r[1] for (e, m), r in thr44.items()}))

,HHI 2000-2015,THEIL 2000-2015,THEIL_PROD 2000-2015,HHI 2000-2022,THEIL 2000-2022,THEIL_PROD 2000-2022
HHI_r1,-30.3172***,,,0.3367,,
,(11.6819),,,(17.9145),,
CAP_r1,0.1537***,0.2428***,0.2590***,0.1105***,0.1984***,0.1865***
,(0.0382),(0.0317),(0.0294),(0.0277),(0.0231),(0.0192)
FDI_r1,0.2371***,0.0870,0.0628,0.2420***,0.0033,0.0044
,(0.0577),(0.0728),(0.0561),(0.0692),(0.0401),(0.0380)
GOV_r1,-0.0825,-0.1053**,-0.0572,-0.0845,-0.0598,-0.0446
,(0.0543),(0.0508),(0.0541),(0.0763),(0.0473),(0.0456)
LAB_r1,-0.1019,-0.2152,-0.3394**,0.0508,-0.0449,0.0510
,(0.0723),(0.1505),(0.1437),(0.1183),(0.1333),(0.1016)


`_r1` là hệ số khi EDI ≤ γ (regime 1, đa dạng hóa cao), `_r2` khi EDI > γ (regime 2). Bài báo: regime 1 là HHI < 0,060 / THE < 2,069.


In [23]:
luu("10c_BaiBao_BacHai", bang({f"{e} {m}": fit(d, [e, f"{e}2", *CONTROLS, "GFC"]) for m, d in MAU44.items() for e in EDIS44}))

,HHI 2000-2015,THEIL 2000-2015,THEIL_PROD 2000-2015,HHI 2000-2022,THEIL 2000-2022,THEIL_PROD 2000-2022
HHI,-2.2488,,,-1.7642,,
,(6.5114),,,(5.8726),,
HHI2,-8.8860,,,-3.3888,,
,(9.5811),,,(8.2669),,
CAP,0.1731***,0.1820***,0.1801***,0.1544***,0.1601***,0.1621***
,(0.0571),(0.0588),(0.0564),(0.0514),(0.0517),(0.0487)
FDI,0.0810,0.0767,0.0810,0.0182,0.0194,0.0222
,(0.0550),(0.0527),(0.0550),(0.0450),(0.0450),(0.0481)
GOV,-0.3886***,-0.3593***,-0.3657***,-0.4729***,-0.4514***,-0.4582***
,(0.1182),(0.1108),(0.1011),(0.0932),(0.0908),(0.0875)


### Bảng trình bày theo bố cục bài báo

Bảng 2–5 giống bố cục Trinh & Thuy (2021), mẫu 42 nước của bài báo, giai đoạn 2000–2015. Ghi ra `data/output/KLTN_bang_bai_bao.xlsx` (font Times New Roman, kẻ khung kiểu bài báo) để dán vào Word. Bảng 5 dùng FE với sai số chuẩn cluster theo quốc gia; kiểm định Poolability (F) và Hausman dùng hiệp phương sai thường.


In [24]:
from openpyxl.styles import Alignment, Border, Font, Side

M, KY = MAU44["2000-2015"], "2000-2015"
VARS44 = [Y, *EDIS44, *CONTROLS]
hs = lambda r, v: f"{r.params[v]:.3f}{stars(r.pvalues[v])}" if v in r.params.index else ""

bang2 = pd.concat([M[VARS44].mean(), M[VARS44].median(), M[VARS44].max(), M[VARS44].min(), M[VARS44].std(),
                   M[VARS44].skew(), M[VARS44].kurt() + 3], axis=1).T.round(3)  # kurtosis thô (phân phối chuẩn = 3) như EViews
bang2.index = ["Mean", "Median", "Maximum", "Minimum", "Standard deviation", "Skewness", "Kurtosis"]

bang3 = pd.DataFrame({e: {"0 vs 1 (F-stat)": f"{thr44[e, KY][0]['F']:.3f}{sao_bp(thr44[e, KY][0]['F'])}",
                          "Giá trị tới hạn Bai–Perron 5% / 1%": f"{CV_BP[0.05]:.3f} / {CV_BP[0.01]:.3f}",
                          "p-value (bootstrap)": f"{thr44[e, KY][0]['p bootstrap']:.3f}",
                          "Threshold value": f"{thr44[e, KY][0]['Ngưỡng (γ)']:.3f}"} for e in EDIS44})

bang4 = {}
for e in EDIS44:
    t, r = thr44[e, KY]; g = t["Ngưỡng (γ)"]
    k = len(r.params) - 1
    chung = {"C": hs(r, "const"), "GFC": hs(r, "GFC"), "R2": f"{r.rsquared:.3f}",
             "Adj. R2": f"{1 - (1 - r.rsquared) * (r.nobs - 1) / (r.nobs - k - 1):.3f}",
             "F-stat (Prob)": f"{r.f_statistic.stat:.3f} ({r.f_statistic.pval:.3f})", "Số quan sát": f"{int(r.nobs)}"}
    for j, dau in [(1, "≤"), (2, ">")]:
        bang4[f"{e} {dau} {g:.3f}"] = {"EDI": hs(r, f"{e}_r{j}"), **{v: hs(r, f"{v}_r{j}") for v in CONTROLS},
                                       **(chung if j == 1 else dict.fromkeys(chung, ""))}
bang4 = pd.DataFrame(bang4)

bang5 = {}
for e in EDIS44:
    xs = [e, f"{e}2", *CONTROLS, "GFC"]
    r, (h, ph) = fit(M, xs), hausman(M, xs)
    f = fit(M, xs, cov="unadjusted").f_pooled
    bang5[e] = {"EDI": hs(r, e), "EDI²": hs(r, f"{e}2"), **{v: hs(r, v) for v in [*CONTROLS, "GFC"]},
                "R2 (within)": f"{r.rsquared:.3f}", "Số quan sát": f"{int(r.nobs)}",
                "Poolability test (F-stat)": f"{f.stat:.3f} ({f.pval:.3f})", "Hausman test (Chi-sq stat)": f"{h:.3f} ({ph:.3f})"}
bang5 = pd.DataFrame(bang5)

GHI_CHU = "Ghi chú: ***, **, * là mức ý nghĩa 1%, 5%, 10% (Bảng 3: theo giá trị tới hạn Bai–Perron như bài báo; p-value bootstrap theo cụm quốc gia báo cáo kèm). Mẫu 42 nước trong Bảng 1 của Trinh & Thuy (2021), 2000–2015. Nguồn: tác giả tính toán."
BANG = {"Bang2_ThongKe": ("Bảng 2. Thống kê mô tả", bang2, False),
        "Bang3_KiemDinhNguong": ("Bảng 3. Kết quả kiểm định ngưỡng", bang3, False),
        "Bang4_HoiQuyNguong": ("Bảng 4. Kết quả hồi quy ngưỡng (biến phụ thuộc: GDPG)", bang4, True),
        "Bang5_BacHaiFE": ("Bảng 5. Kiểm định độ vững bằng mô hình bậc hai (Fixed Effects)", bang5, False)}

FONT, MANH, DAY = "Times New Roman", Side(style="thin"), Side(style="medium")
with pd.ExcelWriter(OUTPUT_BAIBAO, engine="openpyxl") as w:
    for ten, (tieu_de, b, gop) in BANG.items():
        b.to_excel(w, sheet_name=ten, startrow=1)
        ws = w.sheets[ten]
        cuoi, ncot = len(b) + 2, b.shape[1] + 1
        ws["A1"], ws.cell(cuoi + 1, 1).value = tieu_de, GHI_CHU
        for row in ws.iter_rows(min_row=1, max_row=cuoi + 1, max_col=ncot):
            for c in row:
                c.font = Font(name=FONT, size=12, bold=c.row in (1, 2), italic=c.row == cuoi + 1)
                c.alignment = Alignment(horizontal="left" if c.column == 1 or c.row in (1, cuoi + 1) else "center")
                c.border = Border(top=DAY if c.row == 2 else None,
                                  bottom=MANH if c.row == 2 else DAY if c.row == cuoi else None)
        if gop:  # các dòng dùng chung cho 2 regime: gộp ô như bài báo
            for i, v in enumerate(b.index, start=3):
                if v in chung:
                    for c in range(2, ncot, 2):
                        ws.merge_cells(start_row=i, start_column=c, end_row=i, end_column=c + 1)
        ws.column_dimensions["A"].width = 30
        for c in range(2, ncot + 1):
            ws.column_dimensions[ws.cell(2, c).column_letter].width = 18
print(f"Đã lưu {len(BANG)} bảng vào {OUTPUT_BAIBAO}")
bang4


Đã lưu 4 bảng vào C:\2_Company\GSOFT\thaian\kltn-gdp-analysis\data\output\KLTN_bang_bai_bao.xlsx


,HHI ≤ 0.084,HHI > 0.084,THEIL ≤ 5.064,THEIL > 5.064,THEIL_PROD ≤ 1.579,THEIL_PROD > 1.579
EDI,-30.317***,-5.281***,-0.769**,-0.135,-3.308***,0.020
CAP,0.154***,0.134**,0.243***,0.067**,0.259***,0.078**
FDI,0.237***,0.026,0.087,0.160*,0.063,0.187**
GOV,-0.082,-0.088**,-0.105**,-0.042,-0.057,-0.120**
LAB,-0.102,-0.208**,-0.215,-0.086,-0.339**,-0.077
C,2.481,,2.755,,2.551*,
GFC,-2.508***,,-2.451***,,-2.371***,
R2,0.277,,0.272,,0.279,
Adj. R2,0.265,,0.260,,0.266,
F-stat (Prob),23.042 (0.000),,22.412 (0.000),,23.162 (0.000),


## Bước 11. Đặc tả thay thế (chọn trước, báo cáo đầy đủ)

Bốn đặc tả có lý do phương pháp, chạy cho cả HHI và THEIL, báo cáo hết dù ra kết quả nào:
- **Developing + GFC**: đúng tổng thể (nước đang phát triển) và đặc tả của bài mẫu (dummy GFC trong mọi mô hình).
- **TB 5 năm**: trung bình 5 năm (2000–04, …, 2020–22) để lọc biến động chu kỳ, cách làm chuẩn trong nghiên cứu tăng trưởng; chạy cho toàn mẫu và nhóm Developing.
- **EDI trễ 1 năm**: giảm nội sinh (tăng trưởng năm t không tác động ngược lên EDI năm t−1).

Mỗi đặc tả gồm FE tuyến tính, FE bậc hai và threshold Hansen (p-value bootstrap), sai số chuẩn cluster theo quốc gia.

In [25]:
def tb5nam(data):
    """Trung bình 5 năm (2000-04, 05-09, 10-14, 15-19, 20-22); GFC, COVID thành tỷ lệ năm khủng hoảng."""
    d = data.assign(Year=2000 + (data.Year - 2000) // 5 * 5)
    return d.groupby(["ID", "Code", "Year"], as_index=False)[[Y, *EDIS, *CONTROLS, "GFC", "COVID", "DEV"]].mean()


def tre1nam(data):
    """EDI lấy giá trị năm trước (t-1), bỏ năm đầu."""
    d = data.copy()
    d[EDIS] = d.groupby("ID")[EDIS].shift(1)
    return d.dropna(subset=EDIS)


DACTA = {  # tên -> (dữ liệu, biến kiểm soát)
    "Gốc (128 nước, theo năm)": (df, CONTROLS),
    "Developing + GFC": (df[df.DEV == 0], [*CONTROLS, "GFC"]),
    "TB 5 năm (128 nước)": (tb5nam(df), CONTROLS),
    "TB 5 năm, Developing": (tb5nam(df[df.DEV == 0]), CONTROLS),
    "EDI trễ 1 năm (128 nước)": (tre1nam(df), CONTROLS),
}

hang = {}
for ten, (d, ctrl) in DACTA.items():
    for e in EDIS:
        d = d.assign(**{f"{e}2": d[e] ** 2})
        li, qu = fit(d, [e, *ctrl]), fit(d, [e, f"{e}2", *ctrl])
        tt, rg = threshold(d, e, ctrl)
        b1, b2 = qu.params[e], qu.params[f"{e}2"]
        tp = -b1 / (2 * b2)
        hs = lambda r, v: f"{r.params[v]:.3f}{stars(r.pvalues[v])}"
        hang[(ten, e)] = {
            "Số quan sát": int(li.nobs), "Tuyến tính": hs(li, e),
            "β1 (EDI)": hs(qu, e), "β2 (EDI²)": hs(qu, f"{e}2"),
            "Điểm ngoặt": f"{tp:.3f}" + ("" if d[e].min() < tp < d[e].max() else " (ngoài dữ liệu)"),
            "Ngưỡng γ": f"{tt['Ngưỡng (γ)']:.3f}", "F ngưỡng": f"{tt['F']:.2f}", "p bootstrap": f"{tt['p bootstrap']:.3f}",
            "EDI ≤ γ": hs(rg, f"{e}_duoi_nguong"), "EDI > γ": hs(rg, f"{e}_tren_nguong"),
        }
luu("11_DacTaThayThe", pd.DataFrame(hang).T)

Số quan sát Tuyến tính  β1 (EDI)  β2 (EDI²)              Điểm ngoặt Ngưỡng γ F ngưỡng p bootstrap  EDI ≤ γ   EDI > γ
Gốc (128 nước, theo năm) HHI          2944   -3.101**    -1.487     -2.279  -0.326 (ngoài dữ liệu)    0.289     8.71       0.436    2.927    -1.967
                         THEIL        2944      0.388  4.004***  -0.264***                   7.589    7.690     5.89       0.892  0.696**   0.552**
Developing + GFC         HHI          2093    -2.915*    -1.914     -1.400  -0.683 (ngoài dữ liệu)    0.287     4.88       0.787    2.087    -1.906
                         THEIL        2093      0.189     1.821     -0.114                   8.008    7.043     9.81       0.331   -0.374    -0.159
TB 5 năm (128 nước)      HHI           640   -3.978**    -2.566     -1.987  -0.646 (ngoài dữ liệu)    0.415     4.06       0.887   -0.702  -3.752**
                         THEIL         640      0.284   2.838**   -0.190**                   7.459    7.808     6.80       0.728    0.634     0.443
TB 5 năm, Developing     HHI           455    -3.383*    -0.579     -3.895  -0.074 (ngoài dữ liệu)    0.349     4.24       0.827    1.215    -2.437
                         THEIL         455      0.164     0.918     -0.053                   8.581    7.069    13.20       0.101   -0.647    -0.346
EDI trễ 1 năm (128 nước) HHI          2816    -3.160*     0.374     -4.994                   0.037    0.482     5.54       0.813   -0.316   -3.384*
                         THEIL        2816      0.059   3.051**   -0.218**                   7.006    4.742    10.16       0.478   -0.487    -0.129

## Tóm tắt để trả lời 5 câu hỏi (phần B của checklist)

In [26]:
def tom_tat(e):
    l, q, t, i = lin[e, "FE"], quad[e], thr[e][0], inter[e]
    return {
        "1. Hệ số EDI (FE tuyến tính)": f"{l.params[e]:.4f}{stars(l.pvalues[e])} (p={l.pvalues[e]:.3f})",
        "2. Chiều tác động": ("Tập trung cao làm giảm tăng trưởng" if l.params[e] < 0 else "Tập trung cao đi kèm tăng trưởng cao hơn")
                             + ("" if l.pvalues[e] < 0.05 else " (không có ý nghĩa 5%)"),
        "3. Phi tuyến (EDI²)": f"{q.params[e + '2']:.4f}{stars(q.pvalues[e + '2'])}, dạng {'U' if q.params[e + '2'] > 0 else 'U ngược'}, "
                               f"điểm ngoặt {-q.params[e] / (2 * q.params[e + '2']):.4f}",
        "4. Ngưỡng": f"γ={t['Ngưỡng (γ)']:.4f}, p bootstrap={t['p bootstrap']:.3f}",
        "5. Khác biệt Developed (EDIxDEV)": f"{i.params[e + 'xDEV']:.4f}{stars(i.pvalues[e + 'xDEV'])} (p={i.pvalues[e + 'xDEV']:.3f})",
    }

pd.set_option("display.max_colwidth", None)
luu("0_TomTat", pd.DataFrame({e: tom_tat(e) for e in EDIS}))

,HHI,THEIL
1. Hệ số EDI (FE tuyến tính),-3.1006** (p=0.048),0.3877 (p=0.155)
2. Chiều tác động,Tập trung cao làm giảm tăng trưởng,Tập trung cao đi kèm tăng trưởng cao hơn (không có ý nghĩa 5%)
3. Phi tuyến (EDI²),"-2.2792, dạng U ngược, điểm ngoặt -0.3262","-0.2638***, dạng U ngược, điểm ngoặt 7.5891"
4. Ngưỡng,"γ=0.2890, p bootstrap=0.436","γ=7.6900, p bootstrap=0.892"
5. Khác biệt Developed (EDIxDEV),17.4234** (p=0.017),2.0791*** (p=0.008)


In [27]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT) as w:
    for ten in sorted(TABLES):
        TABLES[ten].to_excel(w, sheet_name=ten[:31])
print(f"Đã lưu {len(TABLES)} bảng vào {OUTPUT}")

Đã lưu 21 bảng vào C:\2_Company\GSOFT\thaian\kltn-gdp-analysis\data\output\KLTN_ket_qua.xlsx


## Nhận xét: dữ liệu và kết quả so với bài nghiên cứu mẫu

Bài mẫu: Trinh & Thuy (2021), *Export diversification and economic growth: A threshold regression approach for emerging markets and developing countries*, EJEM 13(2) (`target/document.pdf`). Số dưới đây lấy từ lần chạy với `N_BOOT = 1000`, `SEED = 2024`.

### 1. Dữ liệu: giống và khác bài mẫu

| Nội dung | Bài mẫu | Dữ liệu KLTN | Nhận xét |
|---|---|---|---|
| Mẫu | 44 nước mới nổi và đang phát triển, 1995–2015 | 128 nước (37 phát triển, 91 đang phát triển), 2000–2022, panel cân bằng 2.944 quan sát | Mẫu rộng hơn, có cả nước phát triển. Nhóm Developing (Bước 7) và mẫu 42 nước (Bước 10) là phần so sánh được với bài mẫu. |
| GDPG | Tăng trưởng GDP bình quân đầu người: TB 3,05; SD 3,52; [−14,4; 23,1] | Cùng định nghĩa (WDI NY.GDP.PCAP.KD.ZG): TB 2,30; SD 4,62; [−55,3; 49,1] | Biên độ rộng hơn do các cú sốc lớn; 40 quan sát (1,4%) lệch quá 3 SD. |
| HHI | Tập trung **thị trường** xuất khẩu, TB 0,124, [0; 0,71] | Tập trung thị trường xuất khẩu (World Bank WITS): TB 0,154, [0,03; 0,81]; mẫu 42 nước 2000–2015: TB 0,121, [0,03; 0,71] | ✅ **Cùng khái niệm**, thang đo gần như trùng bài mẫu. |
| THEIL | Overall Theil của IMF, TB 2,94, [1,60; 5,59] | Overall Theil của UNCTAD, TB 6,03, [2,99; 10,39] | Khác nguồn và thang đo, nên ngưỡng 2,069 của bài mẫu không so trực tiếp được. THEIL_PROD gần thang đo bài mẫu hơn. |
| Biến kiểm soát | CAP, FDI, GOV, LAB; dummy AFC 1997 và GFC 2008 | CAP, FDI, GOV, LAB; GFC, COVID ở Bước 8 và Bước 10 | Không có AFC vì mẫu bắt đầu từ 2000. FDI có ngoại lai lớn ([−296; 452]% GDP). |
| Phương pháp ngưỡng | Giá trị tới hạn Bai–Perron | Hansen (1999), p-value bootstrap theo cụm quốc gia | Cách bootstrap thận trọng hơn: cùng mức F nhưng p-value cao hơn. |

### 2. Kết quả toàn mẫu 128 nước

| Câu hỏi | Bài mẫu | KLTN | Khớp? |
|---|---|---|---|
| Chọn mô hình | FE (Hausman p < 0,05) | FE (Hausman 131,2 và 120,3; F-test, LM đều p ≈ 0) | ✅ |
| Tác động tuyến tính (FE) | Không báo cáo | HHI **−3,10\*\*** (p = 0,048); THEIL 0,39 (không ý nghĩa) | ✅ HHI cùng chiều: tập trung thị trường làm giảm tăng trưởng |
| Có ngưỡng không | Có (HHI F = 6,47\*\*\*) | Không: HHI F = 8,71, p = 0,44; THEIL F = 5,89, p = 0,89 | ❌ |
| Bậc hai (FE) | EDI và EDI² đều âm có ý nghĩa | HHI, HHI² không ý nghĩa; THEIL 4,00\*\*\*, THEIL² −0,26\*\*\* (U ngược, điểm ngoặt 7,59) | ❌ HHI; THEIL ngược dấu β₁ |
| Nhóm nước | Chỉ có nước đang phát triển | Developing: HHI −2,75\*; Developed: HHI 11,62\*; HHI×DEV 17,42\*\* | ✅ Tác động âm nằm ở nhóm đang phát triển |
| Khủng hoảng GFC | C08 −2,29\*\*\* | GFC −3,29\*\*\*; HHI×GFC 4,11\*\* | ✅ |
| CAP, GOV, LAB | CAP +\*\*\*, GOV −\*\*\*, LAB + | CAP 0,12\*\*\*, GOV −0,40\*\*\*, LAB 0,27\*\*\* | ✅ |
| FDI | Dương có ý nghĩa | Toàn mẫu không ý nghĩa; Developing 0,18\*\*\*; sau winsorize 0,05\*\* | ✅ khi bỏ ngoại lai |
| Độ vững (winsorize 1%–99%) | | HHI −2,31\* | ✅ cùng dấu, mức ý nghĩa giảm nhẹ |

### 3. Tái lập trên 42/44 nước của bài báo (Bước 10)

Phương trình (2) của bài báo (pooled, mọi biến đổi hệ số theo regime, hằng số và GFC chung); thiếu Guyana, Jamaica. Giá trị tới hạn Bai–Perron (trimming 15%, q = 5, quy về F chia q): 3,23 / 3,65 / 4,48 ở mức 10% / 5% / 1%.

| | Bài báo (1995–2015) | KLTN 2000–2015 | KLTN 2000–2022 |
|---|---|---|---|
| Ngưỡng HHI, theo Bai–Perron (như bài báo) | Có, γ = 0,060, F = 6,47\*\*\* | ✅ **Có**: γ = 0,084, F = 5,20\*\*\* | ✅ Có: γ = 0,064, F = 5,48\*\*\* |
| Ngưỡng HHI, theo bootstrap cụm | | Không (p = 0,16) | Không (p = 0,30) |
| Hệ số HHI theo regime | Âm cả 2 regime; regime 2 −5,27\*\* | ✅ **Âm cả 2 regime**: −30,32\*\*\* và **−5,28\*\*\*** | Regime 2 −4,40\*\*\* |
| Ngưỡng Theil, theo Bai–Perron | Có, F = 4,78\*\* | ✅ THEIL F = 7,06\*\*\*; THEIL_PROD F = 9,28\*\*\* | ✅ THEIL 9,45\*\*\*; THEIL_PROD 12,77\*\*\* |
| Ngưỡng Theil, theo bootstrap cụm | | THEIL p = 0,09; ✅ THEIL_PROD p = 0,02 | ✅ THEIL p = 0,03; THEIL_PROD p = 0,01 |
| Hệ số Theil | Âm, có ý nghĩa ở regime 2 | Âm, có ý nghĩa ở regime 1 (THEIL −0,77\*\*, THEIL_PROD −3,31\*\*\*) | Tương tự |
| GFC | −2,29\*\*\* | ✅ −2,51\*\*\* | ✅ −1,86\*\*\* |
| Bậc hai FE (Bảng 5) | EDI và EDI² đều âm có ý nghĩa | Âm cả hai nhưng không ý nghĩa | Không ý nghĩa |

### 4. Đặc tả thay thế (Bước 11)

| Đặc tả | HHI tuyến tính | HHI bậc hai | Ngưỡng HHI (p) | THEIL bậc hai | Ngưỡng THEIL (p) |
|---|---|---|---|---|---|
| Gốc (128 nước, theo năm) | −3,10\*\* | Không ý nghĩa | 0,44 | U ngược \*\*\* | 0,89 |
| Developing + GFC | −2,92\* | Không ý nghĩa | 0,79 | Không ý nghĩa | 0,33 |
| TB 5 năm (128 nước) | −3,98\*\* | Không ý nghĩa | 0,89 | U ngược \*\* | 0,73 |
| TB 5 năm, Developing | −3,38\* | Không ý nghĩa | 0,83 | Không ý nghĩa | 0,10 |
| EDI trễ 1 năm | −3,16\* | Không ý nghĩa | 0,81 | U ngược \*\* | 0,48 |

Tác động âm tuyến tính của HHI **vững qua cả 5 đặc tả**. Không đặc tả nào cho thấy ngưỡng hay dạng bậc hai có ý nghĩa với HHI. Dạng U ngược của THEIL chỉ có ở toàn mẫu, mất đi khi xét riêng nhóm Developing.

### 5. Kết luận

1. Với HHI thị trường (cùng khái niệm bài mẫu), **đa dạng hóa thị trường xuất khẩu hỗ trợ tăng trưởng**: hệ số FE âm có ý nghĩa trên toàn mẫu, rõ ở nhóm đang phát triển, và vững khi winsorize.
2. Trên mẫu và giai đoạn của bài báo, hệ số HHI âm ở cả hai regime, và regime trên ngưỡng (−5,28\*\*\*) gần như trùng bài báo (−5,27\*\*). Theo giá trị tới hạn Bai–Perron như bài báo, **cả HHI, THEIL, THEIL_PROD đều có ngưỡng ở mức 1%**, tái lập kết luận của bài báo. Kiểm định bootstrap theo cụm quốc gia (chặt hơn vì tính đến tương quan trong cùng quốc gia) chỉ xác nhận ngưỡng của THEIL_PROD (cả hai giai đoạn) và THEIL (2000–2022); với HHI thì không (p = 0,16). Nên báo cáo cả hai và nói rõ khác biệt.
3. Tác động phi tuyến của HHI **không được xác nhận**: không có ngưỡng, không có dạng bậc hai ở cả 5 đặc tả (Bước 11). Bằng chứng phi tuyến chỉ có ở Theil: ngưỡng của THEIL_PROD trên mẫu 42 nước, và dạng U ngược của THEIL trên toàn mẫu. Đây là kết quả nên viết trung thực trong chương thảo luận.
4. Phân loại DEV của 43 nước bổ sung đang là giả định, cần đối chiếu lại trước khi viết bản cuối.